# Pharmasense — Prophet Baseline Forecaster
**Model 5 of 7 — Statistical baseline + fallback forecaster**

## What this model does
Prophet is Facebook/Meta's open-source time-series forecasting library, designed for business time series with:
- **Trend** (linear or logistic growth)
- **Yearly seasonality** (Fourier series — captures monsoon + winter cycles automatically)
- **Weekly seasonality** (day-of-week effects)
- **Holiday effects** (we add India national health campaigns)

## Why we need it
1. **It was planned in `stack.md` from day one** — listed as `Prophet (fallback forecaster if model artifacts missing)`
2. **Fallback:** If TFT/N-BEATS artifacts are unavailable at inference, Prophet provides a solid statistical estimate
3. **Ensemble member:** Prophet's decomposition (trend + seasonal components) feeds the `DemandEnsemble` combiner
4. **Interpretability:** Decomposition plots show judges exactly why demand spikes (monsoon → dengue → ORS)
5. **No GPU needed:** Runs in ~5 minutes on Colab CPU

## What we train on
| Input | Source |
|---|---|
| All 11 condition types | `ncd_demand_weekly` (2,580,666 rows) |
| 641 districts × 11 conditions | One Prophet model per condition type (not per district — share parameters) |
| 2018-W01 → 2023-W52 (train) | 6 years |
| 2024-W01 → 2024-W52 (test) | 1 year holdout |

## Before running
1. **Runtime → Change runtime type → CPU** (no GPU needed)
2. **Add Colab Secret:** 🔑 sidebar → `PHARMASENSE_DB_PASSWORD`
3. **Runtime → Run All**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhi-dev99/agtechathon/blob/main/ml/notebooks/05_train_prophet.ipynb)

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'prophet', 'psycopg2-binary', 'python-dotenv', 'plotly'], check=True)
print('[OK] Dependencies installed')

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────────
import os
from google.colab import userdata

DB_CONFIG = {
    # Transaction Pooler — required for Colab (IPv6 compatible)
    'host':     'aws-1-ap-south-1.pooler.supabase.com',
    'port':     6543,
    'dbname':   'postgres',
    'user':     'postgres.ylbtbfccijamkzdnwtya',  # project ref in user field
    'password': userdata.get('PHARMASENSE_DB_PASSWORD'),
    'sslmode':  'require',
}

TRAIN_CUTOFF = '2023-12-31'   # train on 2018-2023
TEST_CUTOFF  = '2024-12-31'   # test on 2024
FORECAST_HORIZON_WEEKS = 8    # 8-week forecast output
ARTIFACTS_DIR = '/content/drive/MyDrive/pharmasense_artifacts/'

import os
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
print(f'[OK] Config ready. Artifacts → {ARTIFACTS_DIR}')

In [ ]:
# ── Mount Drive (to persist model artifacts) ───────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('[OK] Google Drive mounted')

In [ ]:
# ── Load data from Supabase ─────────────────────────────────────────────────────
import psycopg2
import pandas as pd
import numpy as np

def get_conn():
    return psycopg2.connect(**DB_CONFIG, connect_timeout=30)

print('Loading ncd_demand_weekly from Supabase...')
conn = get_conn()

df = pd.read_sql("""
    SELECT
        district_id,
        ncd_type       AS condition,
        week_start_date::date AS ds,
        demand_units   AS y,
        patients_est,
        district_pop
    FROM ncd_demand_weekly
    ORDER BY ncd_type, district_id, week_start_date
""", conn)

conn.close()
df['ds'] = pd.to_datetime(df['ds'])

print(f'[OK] Loaded {len(df):,} rows')
print(f'     Conditions: {sorted(df.condition.unique())}')
print(f'     Districts:  {df.district_id.nunique()}')
print(f'     Date range: {df.ds.min().date()} → {df.ds.max().date()}')

In [ ]:
# ── Aggregate to national level per condition ──────────────────────────────────
# Prophet trains on ONE time series per condition (national aggregate).
# At inference, we scale by district_pop / national_pop.
# This avoids training 641×11 = 7,051 separate models.

national = (
    df.groupby(['condition', 'ds'])
    .agg(y=('y', 'sum'), patients=('patients_est', 'sum'))
    .reset_index()
)

NATIONAL_POP = df.groupby('district_id')['district_pop'].first().sum()
print(f'National population: {NATIONAL_POP:,.0f}')
print(f'National-level series shape: {national.shape}')
national.head()

In [ ]:
# ── Define India health campaign holidays (Prophet "changepoints") ─────────────
# These are national events that cause demand spikes:
#   National Deworming Day (NDD): Feb 10 and Aug 10 every year
#   World TB Day (DOTS completion push): March 24
#   National Health Mission quarterly review: Jan/Apr/Jul/Oct (minor)

import itertools
from prophet import Prophet

def make_india_holidays():
    rows = []
    for year in range(2018, 2025):
        rows += [
            {'holiday': 'national_deworming_day_feb', 'ds': pd.Timestamp(f'{year}-02-10'),
             'lower_window': -1, 'upper_window': 7},
            {'holiday': 'national_deworming_day_aug', 'ds': pd.Timestamp(f'{year}-08-10'),
             'lower_window': -1, 'upper_window': 7},
            {'holiday': 'world_tb_day',              'ds': pd.Timestamp(f'{year}-03-24'),
             'lower_window': 0, 'upper_window': 3},
            {'holiday': 'monsoon_onset',             'ds': pd.Timestamp(f'{year}-06-01'),
             'lower_window': 0, 'upper_window': 14},
        ]
    return pd.DataFrame(rows)

holidays = make_india_holidays()
print(f'[OK] {len(holidays)} holiday events defined')
print(holidays.holiday.value_counts())

In [ ]:
# ── Train one Prophet model per condition ─────────────────────────────────────
import pickle
import warnings
warnings.filterwarnings('ignore')

CONDITIONS = sorted(national.condition.unique())
models     = {}
forecasts  = {}
metrics    = {}

print(f'Training {len(CONDITIONS)} Prophet models...')
print('='*60)

for cond in CONDITIONS:
    series = national[national.condition == cond][['ds','y']].copy()
    series = series.sort_values('ds').reset_index(drop=True)

    # Train/test split
    train = series[series.ds <= TRAIN_CUTOFF]
    test  = series[series.ds >  TRAIN_CUTOFF]

    # Prophet model config:
    # - yearly_seasonality=True: Fourier terms auto-detect monsoon cycle
    # - weekly_seasonality=False: weekly data, no within-week pattern
    # - seasonality_mode='multiplicative': demand scales with population trend
    # - changepoint_prior_scale=0.3: allow trend changes (outbreak years)
    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode='multiplicative',
        changepoint_prior_scale=0.3,
        seasonality_prior_scale=10.0,
        holidays=holidays,
        interval_width=0.90,   # 90% prediction interval (matches LP optimizer q90)
    )
    m.fit(train)

    # Forecast on test period + future
    future = m.make_future_dataframe(periods=len(test) + FORECAST_HORIZON_WEEKS, freq='W')
    fc     = m.predict(future)

    # Evaluate on test set
    test_fc = fc[fc.ds.isin(test.ds)][['ds','yhat','yhat_lower','yhat_upper']]
    merged  = test.merge(test_fc, on='ds')
    if len(merged) > 0:
        mae  = np.mean(np.abs(merged.y - merged.yhat))
        mape = np.mean(np.abs((merged.y - merged.yhat) / (merged.y + 1))) * 100
        # Coverage: fraction of actuals inside 90% prediction interval
        cov  = np.mean((merged.y >= merged.yhat_lower) & (merged.y <= merged.yhat_upper)) * 100
    else:
        mae, mape, cov = float('nan'), float('nan'), float('nan')

    models[cond]    = m
    forecasts[cond] = fc
    metrics[cond]   = {'mae': mae, 'mape_pct': mape, 'coverage_pct': cov}

    print(f'  {cond:<35} MAE={mae:>12,.0f}  MAPE={mape:>5.1f}%  PI90_cov={cov:>5.1f}%')

print('='*60)
print('[OK] All Prophet models trained')

In [ ]:
# ── Plot decomposition for 3 representative conditions ────────────────────────
import matplotlib.pyplot as plt

PLOT_CONDITIONS = ['viral_fever_urti', 'helminth_infestation', 'hypertension']

for cond in PLOT_CONDITIONS:
    if cond not in models:
        continue
    m  = models[cond]
    fc = forecasts[cond]
    print(f'\n--- {cond} ---')
    fig = m.plot(fc, figsize=(14, 4))
    fig.suptitle(f'Prophet Forecast — {cond}', fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()

    fig2 = m.plot_components(fc, figsize=(14, 8))
    fig2.suptitle(f'Trend + Seasonality Decomposition — {cond}', fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()
    # KEY INSIGHT: helminth_infestation should show Feb + Aug spikes (NDD)
    # viral_fever_urti should show Jul-Sep monsoon peak + Nov-Jan winter peak
    # hypertension should show smooth upward trend (aging population)

In [ ]:
# ── Summary metrics table ─────────────────────────────────────────────────────
metrics_df = pd.DataFrame(metrics).T
metrics_df.index.name = 'condition'
metrics_df = metrics_df.round(2)
print('\nProphet — Test Set Performance (2024)')
print('='*65)
print(metrics_df.to_string())
print(f'\nMean MAPE across all conditions: {metrics_df.mape_pct.mean():.1f}%')
print(f'Mean PI90 coverage:              {metrics_df.coverage_pct.mean():.1f}%  (target >= 87%)')

In [ ]:
# ── Save model artifacts ──────────────────────────────────────────────────────
import json

# Save all models as a single pickle bundle
bundle_path = os.path.join(ARTIFACTS_DIR, 'prophet_models_v1.pkl')
with open(bundle_path, 'wb') as f:
    pickle.dump({
        'models':       models,
        'national_pop': NATIONAL_POP,
        'train_cutoff': TRAIN_CUTOFF,
        'conditions':   CONDITIONS,
        'metrics':      metrics,
    }, f)
print(f'[OK] Saved Prophet bundle → {bundle_path}')

# Save metrics as JSON for CI/eval pipeline
metrics_path = os.path.join(ARTIFACTS_DIR, 'prophet_metrics_v1.json')
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'[OK] Saved metrics → {metrics_path}')

# Save future forecasts as CSV (for LP optimizer fallback)
future_rows = []
for cond, fc in forecasts.items():
    fc_tail = fc.tail(FORECAST_HORIZON_WEEKS)[['ds','yhat','yhat_lower','yhat_upper']].copy()
    fc_tail['condition'] = cond
    future_rows.append(fc_tail)

future_df = pd.concat(future_rows, ignore_index=True)
fc_path   = os.path.join(ARTIFACTS_DIR, 'prophet_forecast_8wk.csv')
future_df.to_csv(fc_path, index=False)
print(f'[OK] Saved 8-week ahead forecasts → {fc_path}')
print(f'\nArtifacts written:')
print(f'  prophet_models_v1.pkl     — trained Prophet objects (11 models)')
print(f'  prophet_metrics_v1.json   — test set MAE/MAPE/coverage per condition')
print(f'  prophet_forecast_8wk.csv  — 8-week national demand forecast')

## How Prophet integrates with the rest of the system

```
At inference time:

  prophet_models_v1.pkl
         |
         ↓
  m.predict(future_df)    → yhat (national)
         |
         ↓
  scale to district:      yhat_district = yhat_national × (district_pop / national_pop)
         |
         ↓
  DemandPrediction object (ensemble.py)
         |
         ↓
  DemandEnsemble.combine(tft_pred, prophet_pred)   ← use if N-BEATS unavailable
         |
         ↓
  DrugForecast → ProcurementOptimizer → ProcurementOrder
```

Prophet is the **fallback** — if TFT isn't available, Prophet keeps the LP optimizer fed.